In [1]:
import numpy as np
import matplotlib.pyplot as plt


/home/student/Pulpit/valeriia/.venv/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
# Momentum
def P(M, m1, m2):
    p1=np.sqrt(M**2+m1**2-m2**2)/(2*M)
    p2=np.sqrt(M**2+m2**2-m1**2)/(2*M)
    return (p1+p2)/2/M


In [4]:
# Angles and vectors of momentum

def angles():
    phi=np.random.uniform(0, 2*np.pi)
    costheta=np.random.uniform(-1, 1)
    theta=np.arccos(costheta)
    return phi, theta

def momentum_vectors(p, phi, theta):
    px=p*np.sin(theta)*np.cos(phi)
    py=p*np.sin(theta)*np.sin(phi)
    pz=p*np.cos(theta)
    return np.array([px, py, pz])

def generator(M, m1, m2):
    p=P(M, m1, m2)
    theta, phi = angles()
    p1=momentum_vectors(p, phi, theta)
    p2=-p1
    return p1, p2


In [28]:
def beta(momentum, mass):
    E = np.sqrt(momentum**2 + mass**2)
    return momentum / E

def bethe_bloch(momentum, mass):
    a, b, c = 2.0, 4.0, 0.05  
    β = beta(momentum, mass)
    return a / (β**2) * (b - β**2 - np.log(c + 1/β**2))

TPC_dEdx_resolution = 0.05

def measure_dEdx(momentum, mass):
    true = bethe_bloch(momentum, mass)
    smeared = np.random.normal(true, TPC_dEdx_resolution * true)
    return smeared


In [29]:
def nsigma_pid(dEdx_meas, momentum, mass_hypothesis):
    expected = bethe_bloch(momentum, mass_hypothesis)
    sigma = TPC_dEdx_resolution * expected
    return (dEdx_meas - expected) / sigma


In [30]:
def smear_momentum(p_vec):
    p = np.linalg.norm(p_vec)
    sigma_rel = np.sqrt(0.01**2 + (0.01*p)**2)
    p_smeared = np.random.normal(p, sigma_rel*p)
    return p_smeared * p_vec/p if p > 0 else p_vec


In [15]:
# Masses in GeV and definitions of the decays

masses = {'pion': 0.13957, 'muon': 0.10566, 'kaon': 0.49368, 'proton': 0.93827, 'jpsi': 3.0969, 'psi2s': 3.6861}

def jpsi2muon2():
    return generator(masses['jpsi'], masses['muon'], masses['muon'])

def psi2s2muon2():
    return generator(masses['psi2s'], masses['muon'], masses['muon'])

def pp_interaction():
    return generator(masses['proton'], masses['proton'], masses['proton'])



# Event gereration and plotting qa plots of invariant mass distributin and pT distribution

def event_generator(decay, N=10000):
    momentum = []
    mass_list = []
    if decay == "jpsi":
        m1 = m2 = 0.10566
    elif decay == "rho":
        m1 = m2 = 0.13957
    elif decay == "psi2s":
        m1 = m2 = 0.10566
    elif decay == "pp":
        m1 = m2 = 0.93827
    else:
        raise ValueError("Unknown decay")
    for _ in range(N):
        if decay == "rho":
            p1, p2= rho2pi()
            mass =masses['rho']
        elif decay == "jpsi":


In [ ]:
def gg2ll(m_gamma, m_lepton1, m_lepton2):
    return generator(m_gamma, m_lepton1, m_lepton2) 


In [31]:
def qa_plot_momentum(ps, title="Momentum distribution"):
    plt.figure(figsize=(6,4))
    plt.hist(ps, bins=50, histtype='step')
    plt.xlabel("Momentum [GeV/c]")
    plt.ylabel("Counts")
    plt.yscale("log")
    plt.title(title)
    plt.show()


In [ ]:
def qa_plot_dedx(p_list, dedx_list):
    plt.figure(figsize=(8, 6))
    plt.scatter(p_list, dedx_list, s=3)
    plt.xlabel('$p [GeV]$')
    plt.ylabel('dE/dx [a.u.]')
    plt.yscale("log")
    plt.show()


In [33]:
def qa_plot_nsigma(nsig_pi, nsig_p):
    plt.figure(figsize=(6,4))
    plt.hist(nsig_pi, bins=50, alpha=0.6, label="Nσ(π hypothesis)")
    plt.hist(nsig_p, bins=50, alpha=0.6, label="Nσ(p hypothesis)")
    plt.legend()
    plt.xlabel("Nσ")
    plt.yscale("log")
    plt.show()


In [ ]:
def qa_plot_bethe_bloch():
    # Define momentum range (GeV/c)
    p = np.linspace(0.1, 5.0, 400)

    # Particle masses (GeV/c^2)
    masses_bb = {
        'pion': 0.13957,
        'kaon': 0.49367,
        'proton': 0.93827
    }

    plt.figure(figsize=(7,5))

    # Plot Bethe–Bloch curves for π, K, p
    for name, m in masses_bb.items():
        dEdx = bethe_bloch(p, m)
        plt.plot(p, dEdx, label=name)

    plt.title("Bethe–Bloch Curves (TPC-like)")
    plt.xlabel("Momentum p [GeV/c]")
    plt.ylabel("dE/dx [arb. units]")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [34]:
# Event gereration and plotting qa plots of invariant mass distributin and pT distribution

def event_generator(decay, N=10000):
    momentum = []
    invariant_mass = []
    dedx_meas = []
    nsig_pi = []
    nsig_p = []

    for _ in range(N):
        if  decay == "jpsi":
            p1, p2 = jpsi2muon2()
            mass = masses['jpsi']

        elif decay == "psi2s":
            p1, p2 = psi2s2muon2()
            mass = masses['psi2s']

        elif decay == "pp":
            p1, p2 = pp_interaction()
            mass = masses['proton']
            
        elif decay == "gg":
            p1, p2 = gg2ll()

        else:
            raise ValueError("Unknown decay")

        p1_s = smear_momentum(p1)
        p = np.linalg.norm(p1_s)

        # measure TPC dE/dx
        dEdx = measure_dEdx(p, m)

        # PID hypotheses
        nsig_pi.append(nsigma_pid(dEdx, p, masses['pion']))
        nsig_p.append(nsigma_pid(dEdx, p, masses['proton']))

        invariant_mass.append(mass)
        p = np.linalg.norm(p1) 
        momentum.append(p)
        dedx_meas.append(dEdx)

    return np.array(momentum), np.array(invariant_mass), np.array(dedx_meas), np.array(nsig_pi), np.array(nsig_p)


In [35]:
def event_generator_all(N=10000):
    all_momentum = {}
    all_inv_mass = {}

    decays = ["jpsi", "psi2s", "pp"] 

    for decay in decays:
        p, m = event_generator(decay, N)
        all_momentum[decay] = p
        all_inv_mass[decay] = m

    return all_momentum, all_inv_mass


In [ ]:
mom, mass, dedx, nsig_pi, nsig_p = event_generator_all(N=5000)

qa_plot_momentum(mom, "ρ→ππ momentum")
qa_plot_dedx(mom, dedx)
qa_plot_nsigma(nsig_pi, nsig_p)
qa_plot_bethe_bloch()

plt.hist(mass["jpsi"], bins=50)
plt.hist(mass["psi2s"], bins=50)
plt.show()


ValueError: scale < 0